## RAG with WebBased URL data


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['LANGCHAIN_API_KEY']=os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGSMITH_PROJECT']=os.getenv('LANGSMITH_PROJECT')
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
os.environ['LANGCHAIN_TRACING_V2']="true"

In [3]:
# lets load data from the langchain docs
from langchain_community.document_loaders import WebBaseLoader
url_loader=WebBaseLoader("https://docs.langchain.com/")
url_loader

/home/hp/Documents/GenAI/LangChain/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
docs=url_loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/', 'title': 'Home - Docs by LangChain', 'language': 'en'}, page_content='Home - Docs by LangChainSkip to main contentDocs by LangChain home pageHomeSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationDocumentationLangChain is the platform for agent engineering. AI teams at Replit, Clay, Rippling, Cloudflare, Workday, and more trust LangChain’s products to engineer reliable agents.LangSmithLangSmith is a platform that helps AI teams use live production data for continuous testing and improvement. LangSmith provides:ObservabilitySee exactly how your agent thinks and acts with detailed tracing and aggregate trend metrics.Learn moreEvaluationTest and score agent behavior on production data or offline datasets to continuously improve performance.Learn morePrompt EngineeringIterate on prompts with version control, prompt optimization, and collaboration features.Learn moreDeploymentShip your agent in one click, using sc

In [6]:
# Data splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)
text_splitter

In [8]:
split_docs=text_splitter.split_documents(docs)
split_docs

[Document(metadata={'source': 'https://docs.langchain.com/', 'title': 'Home - Docs by LangChain', 'language': 'en'}, page_content='Home - Docs by LangChainSkip to main contentDocs by LangChain home pageHomeSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationDocumentationLangChain is the platform for agent engineering. AI teams at Replit, Clay, Rippling, Cloudflare, Workday, and more trust LangChain’s products to engineer reliable agents.LangSmithLangSmith is a platform that helps AI teams use live production data for continuous testing and improvement. LangSmith provides:ObservabilitySee exactly how your agent thinks and acts with detailed tracing and aggregate trend metrics.Learn moreEvaluationTest and score agent behavior on production data or offline datasets to continuously improve performance.Learn morePrompt EngineeringIterate on prompts with version control, prompt optimization, and collaboration features.Learn moreDeploymentShip your agent in one click, using sc

In [ ]:
#Embed the documents,lets use FAISS for vector Store
# we use Ollama for Embedding, Groq doesnot provide embeddings

from langchain_ollama import OllamaEmbeddings
embeddings=OllamaEmbeddings(model="all-minilm")
embeddings

OllamaEmbeddings(model='all-minilm', validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [11]:
from langchain_community.vectorstores import FAISS
vectorstore=FAISS.from_documents(split_docs,embedding=embeddings)
vectorstore

In [15]:
#lets query the db
query="What is LangSmith?"
response=vectorstore.similarity_search(query=query)
response

[Document(id='a48dbf85-9bd0-4f87-a0f9-5b6d3db1de48', metadata={'source': 'https://docs.langchain.com/', 'title': 'Home - Docs by LangChain', 'language': 'en'}, page_content='Home - Docs by LangChainSkip to main contentDocs by LangChain home pageHomeSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationDocumentationLangChain is the platform for agent engineering. AI teams at Replit, Clay, Rippling, Cloudflare, Workday, and more trust LangChain’s products to engineer reliable agents.LangSmithLangSmith is a platform that helps AI teams use live production data for continuous testing and improvement. LangSmith provides:ObservabilitySee exactly how your agent thinks and acts with detailed tracing and aggregate trend metrics.Learn moreEvaluationTest and score agent behavior on production data or offline datasets to continuously improve performance.Learn morePrompt EngineeringIterate on prompts with version control, prompt optimization, and collaboration features.Learn moreDeplo

In [16]:
response_with_score=vectorstore.similarity_search_with_score(query=query)
response_with_score

[(Document(id='a48dbf85-9bd0-4f87-a0f9-5b6d3db1de48', metadata={'source': 'https://docs.langchain.com/', 'title': 'Home - Docs by LangChain', 'language': 'en'}, page_content='Home - Docs by LangChainSkip to main contentDocs by LangChain home pageHomeSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationDocumentationLangChain is the platform for agent engineering. AI teams at Replit, Clay, Rippling, Cloudflare, Workday, and more trust LangChain’s products to engineer reliable agents.LangSmithLangSmith is a platform that helps AI teams use live production data for continuous testing and improvement. LangSmith provides:ObservabilitySee exactly how your agent thinks and acts with detailed tracing and aggregate trend metrics.Learn moreEvaluationTest and score agent behavior on production data or offline datasets to continuously improve performance.Learn morePrompt EngineeringIterate on prompts with version control, prompt optimization, and collaboration features.Learn moreDepl

In [17]:
##lets feed the vectors to the LLM model and then question iter

from langchain_groq import ChatGroq
llm=ChatGroq(model="openai/gpt-oss-20b")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7fcd7e6dd600>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7fcd7e6ddc60>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [27]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
#from_template - takes only a single string
prompt=ChatPromptTemplate.from_template(
"""
Answer the following questions based only on the provided context:
<context>
{context}
</context>
quesion:{input}
"""
)
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following questions based only on the provided context:\n<context>\n{context}\n</context>\nquesion:{input}\n'), additional_kwargs={})])

In [28]:
# to pass list of documents to llm
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following questions based only on the provided context:\n<context>\n{context}\n</context>\nquesion:{input}\n'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7fcd7e6dd600>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7fcd7e6d

In [29]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"What fruits do I like?",
    "context":[Document(page_content="I like the blue sky and also like to eat strawberry icecream.")]}
)

'You like **strawberry**.'

In [30]:
# lets use the data from  vectordb as the context
retriever=vectorstore.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7fcd7e736350>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following questions based only on the provided context:\n<context>\n{context}\n</context>\nquesion:{input}\n'), additional

In [31]:
# lets ask the retriever chain a question
response_from_chain=retrieval_chain.invoke({"input": "what is langchain?"})
response_from_chain

{'input': 'what is langchain?',
 'context': [Document(id='a48dbf85-9bd0-4f87-a0f9-5b6d3db1de48', metadata={'source': 'https://docs.langchain.com/', 'title': 'Home - Docs by LangChain', 'language': 'en'}, page_content='Home - Docs by LangChainSkip to main contentDocs by LangChain home pageHomeSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationDocumentationLangChain is the platform for agent engineering. AI teams at Replit, Clay, Rippling, Cloudflare, Workday, and more trust LangChain’s products to engineer reliable agents.LangSmithLangSmith is a platform that helps AI teams use live production data for continuous testing and improvement. LangSmith provides:ObservabilitySee exactly how your agent thinks and acts with detailed tracing and aggregate trend metrics.Learn moreEvaluationTest and score agent behavior on production data or offline datasets to continuously improve performance.Learn morePrompt EngineeringIterate on prompts with version control, prompt optimization

In [32]:
response_from_chain['answer']

'**LangChain** is a platform and framework for building and deploying AI agents. It provides:\n\n- Core building blocks for creating agents in Python or TypeScript.\n- Low‑level orchestration, memory handling, and human‑in‑the‑loop support (especially through LangGraph).\n- Open‑source tools (Deep Agents, LangGraph, etc.) that let developers quickly assemble agents that can tackle a wide range of tasks.'

In [ ]:
# we used the create_stuff_documents_chain to group llm and the prompt
# we used the create_retrieval_chain to combine the above chanin and the vectordb as retriever